# House Price Prediction — Exploratory Data Analysis

**Project:** House Price Prediction System  
**Developed by:** Sapna Jabeen

This notebook performs a complete exploratory data analysis (EDA) on the raw house price dataset prior to modeling: structural overview, missing-value analysis, distributions, correlations, and outlier inspection.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('../data/house_prices.csv')
df.head()

## 1. Dataset Overview

In [ ]:
print('Shape:', df.shape)
df.info()

In [ ]:
df.describe().T

## 2. Missing Values

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

if len(missing):
    fig, ax = plt.subplots()
    missing.plot(kind='bar', ax=ax, color='#e07a5f')
    ax.set_title('Missing Values by Column')
    plt.show()

**Interpretation:** A small percentage of missing values are present in a few numerical columns. These are minor and will be imputed using median values during preprocessing, which is robust to the outliers also present in the data.

## 3. Target Variable Distribution

In [ ]:
target_col = 'SalePrice' if 'SalePrice' in df.columns else df.columns[-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[target_col], kde=True, ax=axes[0], color='#2a5298')
axes[0].set_title(f'{target_col} Distribution')
sns.boxplot(x=df[target_col], ax=axes[1], color='#38ef7d')
axes[1].set_title(f'{target_col} Boxplot')
plt.tight_layout()
plt.show()

**Interpretation:** The target variable shows a right-skewed distribution typical of real-world price data, with a small number of high-value outlier properties visible in the boxplot's upper whisker.

## 4. Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap')
plt.show()

**Interpretation:** Square footage and overall quality show the strongest positive correlation with sale price, confirming these as primary value drivers. Distance to the city center shows a negative relationship, consistent with real estate market expectations.

## 5. Feature Distributions

In [ ]:
numeric_features = [c for c in numeric_df.columns if c != target_col]
df[numeric_features].hist(figsize=(14, 10), bins=30, color='#1e3c72')
plt.tight_layout()
plt.show()

## 6. Feature vs Target Scatterplots

In [ ]:
top_features = corr[target_col].abs().sort_values(ascending=False).index[1:5]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat in zip(axes.flatten(), top_features):
    sns.scatterplot(data=df, x=feat, y=target_col, ax=ax, alpha=0.6, color='#1e3c72')
    ax.set_title(f'{feat} vs {target_col}')
plt.tight_layout()
plt.show()

## 7. Pairplot (Top Correlated Features)

In [ ]:
pairplot_cols = list(top_features[:3]) + [target_col]
sns.pairplot(df[pairplot_cols], corner=True)
plt.show()

## Summary of Findings

- The dataset is largely clean, with only minor missing values requiring imputation.
- `SquareFeet` and `OverallQuality` are the strongest positive predictors of price.
- `DistanceToCityCenter_km` shows a negative correlation with price.
- The target variable is right-skewed with a handful of high-value outliers, which the modeling pipeline addresses using IQR-based capping.
- Categorical feature `Neighborhood` carries clear price segmentation and is retained as an encoded feature.

These insights directly informed the feature engineering and model selection decisions used in `src/feature_engineering.py` and `src/train_model.py`.